In [ ]:
import rclpy
import time
from rclpy.node import Node
from rclpy.logging import get_logger
from pathlib import Path

from geometry_msgs.msg import Pose
from moveit_msgs.msg import CollisionObject
from shape_msgs.msg import SolidPrimitive

from moveit.planning import MoveItPy
from moveit_configs_utils import MoveItConfigsBuilder

moveit_config = (
    MoveItConfigsBuilder(robot_name="ur", package_name="ur_moveit_config")
    .robot_description_semantic(Path("srdf") / "ur.srdf.xacro", {"name": "ur10e"})
    .moveit_cpp(Path("config") / "moveit_cpp.yaml")
    .to_moveit_configs()
).to_dict()

ur = MoveItPy(node_name="ur", config_dict=moveit_config)
ur_arm = ur.get_planning_component("ur_manipulator")
scene_monitor = ur.get_planning_scene_monitor()

logger = get_logger("ur")


def add_collision_object(scene_monitor, id, size, pose):
    with scene_monitor.read_write() as scene:
        # Create a collision object
        collision_object = CollisionObject()
        collision_object.id = id
        collision_object.header.frame_id = "base_link"

        # Define the box's dimensions
        box_size = SolidPrimitive()
        box_size.type = SolidPrimitive.BOX
        box_size.dimensions = size

        # Set the box's pose
        box_pose = Pose()
        box_pose.position.x = pose[0]
        box_pose.position.y = pose[1]
        box_pose.position.z = pose[2]

        collision_object.primitives.append(box_size)
        collision_object.primitive_poses.append(box_pose)
        collision_object.operation = CollisionObject.ADD

        # Add the box to the scene
        scene.apply_collision_object(collision_object)
        scene.current_state.update()

In [ ]:
add_collision_object(
    scene_monitor,
    id="box1",
    size=[0.1, 0.1, 0.1],
    pose=[0.5, 0.5, 0.5],
)

In [ ]:
add_collision_object(
    scene_monitor,
    id="box2",
    size=[0.2, 0.2, 0.2],
    pose=[0.5, 0.5, 0.5],
)

In [ ]:
add_collision_object(
    scene_monitor,
    id="box3",
    size=[0.1, 0.1, 0.1],
    pose=[-0.5, -0.5, -0.5],
)

In [ ]:
def plan_and_execute(
    robot,
    planning_component,
    single_plan_parameters=None,
    multi_plan_parameters=None,
):
    """A helper function to plan and execute a motion."""
    # plan to goal
    if multi_plan_parameters is not None:
        plan_result = planning_component.plan(multi_plan_parameters=multi_plan_parameters)
    elif single_plan_parameters is not None:
        plan_result = planning_component.plan(single_plan_parameters=single_plan_parameters)
    else:
        plan_result = planning_component.plan()

    # execute the plan
    if plan_result:
        robot_trajectory = plan_result.trajectory
        robot.execute(robot_trajectory, controllers=[])
    else:
        print("Planning failed")

In [ ]:
ur_arm.set_start_state_to_current_state()

from geometry_msgs.msg import PoseStamped

pose_goal = PoseStamped()
pose_goal.header.frame_id = "base_link"
pose_goal.pose.orientation.w = 1.0
pose_goal.pose.position.x = 0.6
pose_goal.pose.position.y = -0.2
pose_goal.pose.position.z = 0.5
ur_arm.set_goal_state(pose_stamped_msg=pose_goal, pose_link="tool0")

In [ ]:
from moveit.planning import MultiPipelinePlanRequestParameters, PlanRequestParameters

single_plan_parameters = PlanRequestParameters(ur, "pilz_lin")
single_plan_parameters.planner_id = "PTP"
single_plan_parameters.max_velocity_scaling_factor = 0.1
single_plan_parameters.max_acceleration_scaling_factor = 0.1
single_plan_parameters.planning_attempts = 10
single_plan_parameters.planning_time = 10.0

In [ ]:
plan_and_execute(ur, ur_arm, single_plan_parameters=single_plan_parameters)